# 03 Models

PLAN.md steps 3-4: baselines and XGBoost, leave-one-match-out, open play only. Logic in `src/passmodel/models.py`; this notebook reads `results/`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from passmodel.load import REPO_ROOT

RESULTS = REPO_ROOT / "results"
metrics = pd.read_csv(RESULTS / "metrics_per_fold.csv")
predictions = pd.read_csv(RESULTS / "predictions.csv")

## Held-out metrics per fold (fold = test match)
Lower log loss and Brier are better; higher ROC-AUC is better.

In [ ]:
metrics

In [ ]:
metrics.pivot(index="model", columns="fold", values="log_loss")

In [ ]:
pivot = metrics[metrics["fold"] != "mean"].pivot(index="fold", columns="model", values="log_loss")
ax = pivot.plot(marker="o", figsize=(8, 4))
ax.set_ylabel("held-out log loss (lower is better)")
ax.set_title("Every model, every test match: the gap between models is consistent across matches");

## D4 sensitivity: what inner CV would have chosen

In [ ]:
inner = pd.read_csv(RESULTS / "d4_inner_cv.csv")
inner[inner["chosen"]]

In [ ]:
inner.pivot_table(index=["max_depth", "learning_rate", "n_estimators"], columns="fold", values="inner_log_loss").round(4)

## Predicted probabilities by outcome (XGBoost, held out)

In [ ]:
x = predictions[predictions["model"] == "xgboost (all features)"]
for done, colour, label in [(True, "#1f5fbf", "completed"), (False, "#d62728", "failed")]:
    plt.hist(x.loc[x["completed"] == done, "p_complete"], bins=30, alpha=0.5, density=True, color=colour, label=label)
plt.xlabel("predicted chance the pass is completed")
plt.legend()
plt.title("Failed passes get lower predicted completion probabilities");